In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.datasets import KarateClub, CitationFull, Planetoid
from dataclasses import dataclass
from torch_geometric.utils import to_undirected
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.datasets import GraphLandDataset
from torch_geometric.utils import subgraph, add_self_loops
import copy
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

cuda_dev = torch.device("cuda")
cpu_dev = torch.device("cpu")

seed = 42


In [2]:
def build_negative_probs(edge_index: torch.Tensor,
    num_nodes: int, alpha: float = 0.75) -> torch.Tensor:
  """P_n(v) ∝ deg(v)^alpha, normalizada em [0,1]."""
  row, col = edge_index
  # Grafo não-direcionado: conta grau pela soma das duas pontas
  deg = torch.bincount(torch.cat([row, col]), minlength=num_nodes).float()
  weights = deg.clamp(min=1e-12).pow(alpha)   # evita prob=0 p/ nós isolados
  return weights / weights.sum()

def sample_negatives(probs: torch.Tensor,
    batch_size: int, Q: int) -> torch.Tensor:
  """Amostra Q negativos por par positivo → shape [B, Q]."""
  total = batch_size * Q
  # replacement=True necessário quando total > num_nodes
  idx = torch.multinomial(probs, num_samples=total, replacement=True)
  return idx.view(batch_size, Q)

def build_csr(edge_index : torch.Tensor, num_nodes:int, force_undirect = False):
  """Passar edge_index em cpu"""
  if force_undirect:
    edge_index = to_undirected(edge_index.cpu(), num_nodes=num_nodes)
  row, col = edge_index
  order = torch.argsort(row)
  col_sorted = col[order]
  row_ptr = torch.zeros(num_nodes + 1, dtype=torch.long)
  deg = torch.bincount(row, minlength=num_nodes)
  row_ptr[1:] = torch.cumsum(deg, 0)
  return row_ptr, col_sorted

def sample_step(cur : torch.Tensor, row_ptr : torch.Tensor, col_idx : torch.Tensor):
  """cur: [B] → nxt: [B]. Um rand por nó, sem loop Python."""
  start = row_ptr[cur]
  end   = row_ptr[cur + 1]
  deg   = end - start         # grau de cada entrada de cada vertex idx em 'cur'
  deg_safe = deg.clamp(min=1)
  # print("deg",deg_safe)
  off = (torch.rand_like(deg, dtype=torch.float, device=start.device) * deg_safe).long() # offset sorteado
  # print("offset sample:", off)
  # off = off.clamp(max=deg_safe - 1) # offset < deg_safe - 1
  idx = (start + off).clamp(max=col_idx.numel() - 1)
  nxt = col_idx[idx]
  return torch.where(deg == 0, cur, nxt) # se tiver aresta retorna nxt, se não, cur

def random_walks(u_batch : torch.Tensor, row_ptr : torch.Tensor, col_idx : torch.Tensor, walk_length : int, num_walks = 1):
  """[B, num_walks, walk_length+1]"""
  B, W = u_batch.numel(), num_walks
  walks = torch.empty(B, W, walk_length + 1,
    dtype=u_batch.dtype, device=u_batch.device)  # [B, W]
  walks[..., 0] = u_batch.unsqueeze(1)
  flat = u_batch.repeat_interleave(W)      # [B*W]
  for t in range(1, walk_length + 1):       # loop curto, tudo vetorizado dentro
      flat = sample_step(flat, row_ptr, col_idx)
      walks[:, :, t] = flat.view(B, W)
  return walks

def sample_positives(u_batch : torch.Tensor, row_ptr : torch.Tensor, 
    col_idx : torch.Tensor, walk_length=5, context_size=2, num_pos=1):
  """v coocorre com u em walk de comprimento fixo  [B, num_pos]."""
  assert context_size <= walk_length
  walks = random_walks(u_batch, row_ptr, col_idx, walk_length, num_pos)
  offsets = torch.randint(1, context_size + 1,
    (u_batch.numel(), num_pos),
    device=u_batch.device)
  return walks.gather(2, offsets.unsqueeze(-1),).squeeze(-1)

def rebuild_graph_data(data : Data, node_idx : torch.Tensor , edge_index : torch.Tensor):
  data = copy.copy(data)
  data.edge_index = edge_index
  data.x = data.x[node_idx]
  data.y = data.y[node_idx]
  n = node_idx.numel()
  for k in ("train_mask", "val_mask", "test_mask"):
    if hasattr(data, k):
        setattr(data, k, None)

    assert int(data.edge_index.max()) < n, "edge_index OOB — faltou relabel_nodes=True?"
  return data

def aggr_add(h : torch.Tensor, edge_index : torch.Tensor):
  out = torch.zeros(h.size(0), h.size(1), device=h.device, dtype=h.dtype)
  out.index_add_(0, edge_index[1], h[edge_index[0]])
  return out

# def aggr_add(h : torch.Tensor, edge_index : torch.Tensor):
#   src, dst = edge_index
#   h_src, h_dst = h[src], h[dst]



def is_in_cpu(item):
  return torch.device("cpu") != item.device

def aggr_mean(h : torch.Tensor, edge_index : torch.Tensor):
  out = aggr_add(h, edge_index)
  
  deg = torch.zeros(h.size(0),device=h.device,dtype=h.dtype)
  ones = torch.ones(edge_index.size(1),device=out.device,dtype=h.dtype)
  deg.index_add_(0,edge_index[1], ones)
  out = out / deg.clamp(min=1).unsqueeze(-1)
  return out

def get_aggr(aggr):
  if aggr == "add":
        return aggr_add
  elif aggr == "mean":
    return aggr_mean
  
  raise NotImplementedError()

class SageLayer(torch.nn.Module):
  def __init__(self, /,in_feat, out_feat, aggr, normalize, device = None):
    super().__init__()
    self.normalize = normalize
    self.aggr_func = get_aggr(aggr)
    self.aggr = aggr
    self.lin_root = torch.nn.Linear(in_features=in_feat,out_features=out_feat,bias=True, device=device)
    self.lin_neigh = torch.nn.Linear(in_features=in_feat,out_features=out_feat,bias=False, device=device)

  def forward(self, x, edge_index):
    neigh_aggr = self.aggr_func(x, edge_index)
    h = self.lin_root(x) + self.lin_neigh(neigh_aggr)
    # h = F.relu(h)
    if self.normalize:
      h = F.normalize(h, p=2, dim=-1)
    return h

from dataclasses import dataclass

@dataclass
class SageConfig:
    layers: int
    in_feat: int
    out_feat: int
    aggr: str 
    normalize: bool
    normalize_intern: bool

    def build(self) -> torch.nn.ModuleList:
      """Constrói a lista de SageLayer conforme a configuração."""
      modules = []
      for i in range(self.layers):
        in_f = self.in_feat if i == 0 else self.out_feat
        out_f = self.out_feat
        # Só a última camada usa `normalize`; as internas usam `normalize_intern`
        norm = self.normalize if i == self.layers - 1 else self.normalize_intern
        from torch_geometric.nn import SAGEConv
        modules.append(
          SAGEConv(
            in_channels=in_f,
            out_channels=out_f,
            aggr=self.aggr,
            normalize=norm
          )
          # SageLayer(
          #   in_feat=in_f,
          #   out_feat=out_f,
          #   aggr=self.aggr,
          #   normalize=norm,
          # )
        )
      return torch.nn.ModuleList(modules)

class Sage(torch.nn.Module):
  def __init__(self, cfg : SageConfig):
    super().__init__()
    self.sage_layers = cfg.build()
    # self.init_weights()

  def init_weights(self):
    for layer in self.sage_layers:
        torch.nn.init.xavier_uniform_(layer.lin_root.weight)
        torch.nn.init.xavier_uniform_(layer.lin_neigh.weight)
        if layer.lin_root.bias is not None:
            torch.nn.init.zeros_(layer.lin_root.bias)

  def forward(self, h : torch.Tensor, edge_index : torch.Tensor):
    for layer in self.sage_layers:
      h = layer(h,edge_index)
    return h

def move_data(data : Data, device : torch.device):
    data.x = data.x.to(device)
    data.edge_index = data.edge_index.to(device)
    data.y = data.y.to(device)
    return data



def train(model : torch.nn.Module, data_train: Data, data_val : Data, opt : torch.optim.Optimizer, Q,epochs, batch):
  N = data_train.x.size(0)
  if not is_in_cpu(data_train.x):
    move_data(data_train, "cpu")
  data_train.edge_index = add_self_loops(data_train.edge_index)[0]

  device = cpu_dev
  if torch.cuda.is_available():
    device = cuda_dev
  
  row_ptr, col_idx = build_csr(data_train.edge_index, N, force_undirect=True)
  probs = build_negative_probs(data_train.edge_index, N)

  row_ptr = row_ptr.to(device)
  col_idx = col_idx.to(device)
  
  train_idx = torch.argwhere(data_val.train_mask.cpu()).to(device)
  val_idx = torch.argwhere(data_val.test_mask.cpu()).to(device)
  
  data_train = move_data(data_train, device)
  data_val = move_data(data_val, device)
  model = model.to(device)

  eval_every = 10
  def evaluate():
    model.eval()
    with torch.no_grad():
      z = model(data_val.x, data_val.edge_index)
    clf = LogisticRegression(max_iter=500)

    x_train = z[train_idx].squeeze().cpu().numpy()
    y_train = data_val.y[train_idx].cpu().numpy()
    y_train = y_train.reshape((y_train.shape[0],))
    clf.fit(x_train, y_train)

    x_val = z[val_idx].squeeze().cpu().numpy()
    y_val = data_val.y[val_idx].cpu().numpy()
    y_val = y_val.reshape((y_val.shape[0],))
    f1 = f1_score(y_val,
      clf.predict(x_val),
      average="micro")

    model.train()
    return {"f1":f1}

  loss_ema = None
  pbar = tqdm(range(epochs), desc="train", unit="epoch", dynamic_ncols=True)
  model.train()
  for epoch in pbar:
    opt.zero_grad()
    z = model(data_train.x, data_train.edge_index)

    u_batch  = torch.randperm(N, device=z.device)[:batch]
    deg_u = row_ptr[u_batch + 1] - row_ptr[u_batch]
    mask = deg_u > 0
    u_batch = u_batch[mask]
    if mask.sum() == 0:
      continue

    neg_idx = sample_negatives(probs, u_batch.numel(), Q)
    pos_idx = sample_positives(u_batch,row_ptr,col_idx, 
      walk_length=2,context_size=2,num_pos=1).squeeze(1)

    z_u, z_pos = z[u_batch], z[pos_idx]
    z_neg = z[neg_idx]  

    # print("z_u X z_pos (shape):", z_u.shape, " X ", z_pos.shape)
    pos_score = (z_u*z_pos).sum()
    neg_score = torch.bmm(z_neg, z_u.unsqueeze(-1)).squeeze(-1) 
    loss = -F.logsigmoid(pos_score).mean() - Q * F.logsigmoid(-neg_score).mean()
    loss_ema = loss if loss_ema == None else 0.98*loss_ema + 0.02*loss
    loss.backward() 
    postfix = { "loss":loss.item(), "loss_ema":loss_ema.item() }
    
    if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
      stats = evaluate()

      postfix = {**postfix, **stats}

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    pbar.set_postfix(**postfix)

from torch_geometric.utils import subgraph

data = Planetoid("/home/milnner00/projects/gnn_recsys/data","cora").data
print((data.edge_index == to_undirected(data.edge_index)).all())

# name pode ser "hm-categories", "web-topics", "artnet-exp", etc.
# dataset = GraphLandDataset(root="/home/milnner00/projects/gnn_recsys/data", name="web-topics", split="THI")

# train_data = dataset[0]  # grafo de treino
# val_data   = dataset[1]  # grafo de validação (nós não vistos no treino)
# test_data  = dataset[2]  # grafo de teste  (nós não vistos no treino nem na val)

N = data.x.size(0)
N_train = int(N * 0.9)
N_val =  N - N_train #int(N * 0.1)
# N_test = N - N_train - N_val
rgn = torch.Generator().manual_seed(seed)
idx = torch.randperm(N,generator=rgn)

splits = torch.split(idx, [N_train, N_val])

train_idx, val_idx = splits

data.train_mask = torch.zeros(data.x.size(0), dtype=torch.bool)
data.train_mask[train_idx] = 1
data.test_mask = ~(data.train_mask)
data.val_mask = None

train_ei = subgraph(train_idx, data.edge_index, relabel_nodes=True)[0]
val_ei = subgraph(val_idx, data.edge_index, relabel_nodes=True)[0]
# test_ei = subgraph(test_idx,data.edge_index, relabel_nodes=True)[0]


train_data = rebuild_graph_data(data, train_idx, train_ei)
val_data = rebuild_graph_data(data, val_idx, val_ei)
# train_data.edge_index = to_undirected(train_data.edge_index)
# val_data.edge_index = to_undirected(val_data.edge_index)
print(train_data)
print(val_data)
print((train_data.edge_index == to_undirected(train_data.edge_index)).all())

from torch_geometric.nn import GraphSAGE

sage_cfg = SageConfig(1, data.x.size(1),256,"mean",False,False)
model = Sage(sage_cfg)
model_pyg = GraphSAGE(data.x.size(1),256,1,aggr="mean")

opt = torch.optim.Adam(model.parameters(), lr=5e-6, weight_decay=5e-4)
# train(model,train_data,data,opt,1, 10000, 30)

tensor(False)
Data(x=[2437, 1433], edge_index=[2, 8660], y=[2437])
Data(x=[271, 1433], edge_index=[2, 100], y=[271])
tensor(False)


/tmp/ipykernel_628001/3923338285.py:274: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = Planetoid("/home/milnner00/projects/gnn_recsys/data","cora").data


In [3]:
ei = data.edge_index
ei_sym = to_undirected(ei)

# (a) igualdade como conjunto de arestas
s_fwd = set(zip(ei[0].tolist(), ei[1].tolist()))
s_sym = set(zip(ei_sym[0].tolist(), ei_sym[1].tolist()))
print("mesmo conjunto:", s_fwd == s_sym)        # True se já era simétrico

# (b) todo (u,v) tem seu (v,u)?
s_rev = set(zip(ei[1].tolist(), ei[0].tolist()))
print("simétrico:", s_fwd == s_rev)             # True → sem self-loops nem assimetria

# (c) comparação elemento a elemento após canonicalizar
from torch_geometric.utils import sort_edge_index, coalesce
ei_c = coalesce(ei)
ei_s = coalesce(ei_sym)
print("mesmo shape:", ei_c.shape, ei_s.shape)
print("igual após coalesce:", (ei_c == ei_s).all())

mesmo conjunto: True
simétrico: True
mesmo shape: torch.Size([2, 10556]) torch.Size([2, 10556])
igual após coalesce: tensor(True)


In [56]:
# 1) Todo nó local referenciado existe em x
assert data_train.edge_index.max() < data_train.num_nodes, "edge_index OOB"

# 2) Cada aresta local mapeia para uma aresta real no grafo original
mapped = train_idx[data_train.edge_index]               # [2, E_sub] em IDs globais
orig   = set(map(tuple, data.edge_index.t().tolist()))
assert set(map(tuple, mapped.t().tolist())).issubset(orig), "aresta inválida"

# 3) Features batem com o subconjunto
assert torch.equal(data_train.x, data.x[train_idx])

NameError: name 'data_train' is not defined

In [56]:
print(data_train.edge_index)
print(mapped)
train_idx[1415]

tensor([[1742,  738,  798,  ...,  210,  450,  832],
        [1415, 1415, 1415,  ..., 2163, 2163, 2163]])
tensor([[ 633, 1862, 2582,  ...,  165, 1473, 2706],
        [   0,    0,    0,  ..., 2707, 2707, 2707]])


tensor(0)

In [68]:
from torch_geometric.nn import SAGEConv

# 1) mesmo grafo, mesmas features, mesma seed
train_data = train_data.to(cpu_dev)
x = train_data.x.clone()
ei = train_data.edge_index.clone()

# 2) seu layer e o SAGEConv com pesos copiados
layer = SageLayer(x.size(1), 32, "mean", normalize=False)
conv_pyg = SAGEConv(in_channels=x.size(1), out_channels=32,
                    aggr='mean', normalize=False, root_weight=True)
# copie pesos: lin_l <- lin_neigh, lin_r <- lin_root
conv_pyg.lin_l.weight.data = layer.lin_neigh.weight.data
conv_pyg.lin_r.weight.data = layer.lin_root.weight.data

# 3) forward de ambos
with torch.no_grad():
    out_pyg = conv_pyg(x, ei)
    out_usr = layer(x, ei)   # sem normalize, para comparar

print("diff:", (out_pyg - out_usr).abs().max().item())

diff: 0.047545865178108215


In [59]:
h = train_data.x
edge_index = train_data.edge_index
from torch_scatter import scatter_mean, scatter_sum
N = train_data.x.size(0)
ref_sum  = scatter_sum(h[edge_index[0]], edge_index[1], dim=0, dim_size=N)
ref_mean = scatter_mean(h[edge_index[0]], edge_index[1], dim=0, dim_size=N)
print((aggr_add(h, edge_index) - ref_sum).abs().max())   # ~0
print((aggr_mean(h, edge_index) - ref_mean).abs().max()) # ~0

tensor(0., device='cuda:0')
tensor(0., device='cuda:0')


In [15]:

# name pode ser "hm-categories", "web-topics", "artnet-exp", etc.
dataset = GraphLandDataset(root="/home/milnner00/projects/gnn_recsys/data", name="web-topics", split="THI")



Data(x=[1112961, 527], edge_index=[2, 13645122], y=[1112961], snapshot='train', mask=[1112961], x_numerical_mask=[527], x_fraction_mask=[527], x_categorical_mask=[527], cross_snapshot_node_id=[1112961])
Data(x=[1751172, 527], edge_index=[2, 20273356], y=[1751172], snapshot='val', mask=[1751172], x_numerical_mask=[527], x_fraction_mask=[527], x_categorical_mask=[527], cross_snapshot_node_id=[1751172])


tensor(159706)

In [21]:
train_data = dataset[0]  # grafo de treino
val_data   = dataset[1]  # grafo de validação (nós não vistos no treino)
test_data  = dataset[2]  # grafo de teste  (nós não vistos no treino nem na val)
print(train_data)
print(train_data.mask)

print(val_data)
print(val_data.mask)

Data(x=[1112961, 527], edge_index=[2, 13645122], y=[1112961], snapshot='train', mask=[1112961], x_numerical_mask=[527], x_fraction_mask=[527], x_categorical_mask=[527], cross_snapshot_node_id=[1112961])
tensor([False,  True, False,  ...,  True,  True,  True])
Data(x=[1751172, 527], edge_index=[2, 20273356], y=[1751172], snapshot='val', mask=[1751172], x_numerical_mask=[527], x_fraction_mask=[527], x_categorical_mask=[527], cross_snapshot_node_id=[1751172])
tensor([False,  True, False,  ..., False, False, False])


In [12]:
row, col = torch.tensor([0, 0, 1, 1, 2, 3,3]),\
          torch.tensor([1, 2, 2, 3, 3, 0,6])
r_max, c_max = row.max().item(), col.max().item()
N = (r_max if r_max > c_max else c_max) + 1
row + 2*N

batch_ei = torch.cat([
  row,   
  # row + N,
  # row + 2*N,

  col,
  # col + N,
  # col + 2*N
], dim = 0,).view(2, col.size(0))

print(batch_ei.shape)

print(batch_ei)
print()

indptr, indices = build_csr(batch_ei,batch_ei.max().item() + 1)
print(indptr)
print(indices)


torch.Size([2, 7])
tensor([[0, 0, 1, 1, 2, 3, 3],
        [1, 2, 2, 3, 3, 0, 6]])

tensor([ 0,  3,  6,  9, 13, 13, 13, 14])
tensor([1, 2, 3, 0, 2, 3, 0, 1, 3, 0, 1, 2, 6, 3])


In [20]:
u_batch = torch.tensor([0,0,0,3,5])
num_walks = 2
walk_length = 2
context_size = 2

"""→ [B, num_walks, walk_length+1]"""
B, W = u_batch.numel(), num_walks
walks = torch.empty(B, W, walk_length + 1,
  dtype=u_batch.dtype, device=u_batch.device)  # [B, W]
print(walks.shape)
print()
walks[..., 0] = u_batch.unsqueeze(1)
print(walks)
print()
flat = u_batch.repeat_interleave(W)       # [B*W]
print("row_ptr",indptr)
print("col_idx",indices)
print(flat)
for t in range(1, walk_length + 1):       # loop curto, tudo vetorizado dentro
    print("enter",flat)
    flat = sample_step(flat, indptr, indices)
    print("out:",flat)
    walks[:, :, t] = flat.view(B, W)
    

torch.Size([5, 2, 3])

tensor([[[                   0, -1142224136920881663,      403452685529856],
         [                   0, -1148417010752565197,  7061908201604473350]],

        [[                   0, -1146164948175359976, -6332456797172047360],
         [                   0,  7061908201604473344,   166100244928921869]],

        [[                   0,    67569391906980080,   166071267286385230],
         [                   0,    72706309992874242, -1148979745522651122]],

        [[                   3, -1136876215442851583,   144182190522047232],
         [                   3,   144119522771652672,   499969931008246801]],

        [[                   5,   284076424715698384,  4467808460222824960],
         [                   5, -1148416741027606527,  5188189583233322752]]])

row_ptr tensor([ 0,  3,  6,  9, 13, 13, 13, 14])
col_idx tensor([1, 2, 3, 0, 2, 3, 0, 1, 3, 0, 1, 2, 6, 3])
tensor([0, 0, 0, 0, 0, 0, 3, 3, 5, 5])
enter tensor([0, 0, 0, 0, 0, 0, 3, 3, 5, 5])
out:

In [24]:
offsets = torch.randint(
  1, 
  context_size + 1,
  (u_batch.numel(), num_walks),
  device=u_batch.device)

pos = walks.gather(2, offsets.unsqueeze(-1)).squeeze(-1)

print("oofsets:", offsets)
print("offsets_unsq:",offsets.unsqueeze(-1))
print("walks:", walks)
print(pos.shape)
print("pos:", pos.squeeze())

oofsets: tensor([[1, 1],
        [2, 1],
        [1, 1],
        [2, 2],
        [1, 1]])
offsets_unsq: tensor([[[1],
         [1]],

        [[2],
         [1]],

        [[1],
         [1]],

        [[2],
         [2]],

        [[1],
         [1]]])
walks: tensor([[[0, 2, 1],
         [0, 2, 0]],

        [[0, 3, 6],
         [0, 3, 0]],

        [[0, 3, 0],
         [0, 1, 3]],

        [[3, 1, 3],
         [3, 1, 0]],

        [[5, 5, 5],
         [5, 5, 5]]])
torch.Size([5, 2])
pos: tensor([[2, 2],
        [6, 3],
        [3, 1],
        [3, 0],
        [5, 5]])


In [ ]:
v = [1, 2, 2, 2, 3, 4, 5, 5, 5, 5]
v = torch.tensor(v).reshape((1,len(v)))

print("vetor:", v)
print(v.shape)
counts = torch.bincount(v[0], minlength=v.max())
print("counts:",counts)
probs = counts / v.size(1)
print("probs:", probs)
print("sum probs:", probs.sum())

samples = torch.multinomial(probs,10000, replacement=True)
count_p = torch.bincount(samples)

probs_p = count_p / 10000
print("samples:", probs_p )



tensor([[2, 2],
        [6, 3],
        [3, 1],
        [3, 0],
        [5, 5]])


In [ ]:
print(graph.get_summary)
graph.data


<bound method Dataset.get_summary of CoraFull()>


/tmp/ipykernel_2817857/2628678813.py:3: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  graph.data


Data(x=[19793, 8710], edge_index=[2, 126842], y=[19793])

In [ ]:
NeighborLoader()

/tmp/ipykernel_2240843/4038582665.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = graph.data


tensor([[ 0.0248, -0.5383,  0.1465,  0.0194,  0.1435],
        [-0.2784, -0.0875, -0.0515,  0.0566, -0.2488],
        [-0.3132,  0.5139, -0.1732, -0.5692, -0.5419]],
       grad_fn=<PermuteBackward0>)


tensor([[-0.4433,  0.0549, -0.1914, -1.0796, -0.4633],
        [-0.4333, -0.0080, -0.2162, -0.4732, -0.3138],
        [-0.4433,  0.0549, -0.1914, -1.0796, -0.4633]],
       grad_fn=<AddmmBackward0>)

In [16]:
l = list(range(1,11))
print(l)
print()
print(l[:5]) # [1, 2, 3, 4, 5]
print()
print(l[2:6]) # [3, 4, 5, 6]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

[1, 2, 3, 4, 5]

[3, 4, 5, 6]
